In [1]:
from langchain_community.document_loaders import PyMuPDFLoader


pdf_path = "data/Indias_Ancient_history_full.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()
print(f"Pages loaded: {len(documents)}")


/var/folders/1f/dc6nbcvj0yzdh4mpnz5f_3l00000gn/T/ipykernel_87635/2781586893.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/Users/dipyaman/Documents/codes/genAI/RAG_practice/rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages loaded: 422


In [2]:
# Load opeanai api key from environment variable
import os
from dotenv import load_dotenv

load_dotenv()

open_api_key = os.environ.get("OPENAI_API_KEY")

langsmith_api_key = os.environ.get("LANGSMITH_API_KEY")
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "rag_eval_ancient"

In [4]:
print(documents[14].page_content)  # Print the first 500 characters of the first page

1
The Significance of
Ancient Indian History
The study of ancient Indian history is important for several reasons. It tells us
how, when, and where people developed the earliest cultures in India, how they
began undertaking agriculture and stock raising which made life secure and
settled. It shows how the ancient Indians discovered and utilized natural
resources, and how they created the means for their livelihood. We get an idea of
how the ancient inhabitants made arrangements for food, shelter, and transport,
and learn how they took to farming, spinning, weaving, metalworking, and the
like, how they cleared forests, founded villages, cities, and eventually large
kingdoms.
People are not considered civilized unless they know how to write. The
different forms of writing prevalent in India today are all derived from the
ancient scripts. This is also true of the languages that we speak today. The
languages we use have roots in ancient times, and have developed through the
ages.
Unity in 

In [5]:
full_text = " ".join([doc.page_content for doc in documents])
full_text_shorten  = " ".join([doc.page_content for doc in documents[14:]]) # Skip the first 13 pages as they are not relevant to the chapters

In [6]:
import re
pattern = r"(\d+)\.\s*\n([^\n]+)"
matches = list(re.finditer(pattern, full_text))

In [7]:
matches = matches[:33] 


In [8]:
chapters = []

for m in matches:  # Skip the first 33 matches
    print("Chapter:", m.group(1))
    print("Title:", m.group(2))
    print("-" * 50)
    chapters.append(m.group(2))  # Append the chapter title to the list

Chapter: 1
Title: The Significance of Ancient Indian History
--------------------------------------------------
Chapter: 2
Title: Modern Historians of Ancient India
--------------------------------------------------
Chapter: 3
Title: Nature of Sources and Historical Construction
--------------------------------------------------
Chapter: 4
Title: Geographical Setting
--------------------------------------------------
Chapter: 5
Title: Ecology and Environment
--------------------------------------------------
Chapter: 6
Title: The Linguistic Background
--------------------------------------------------
Chapter: 7
Title: Human Evolution: The Old Stone Age
--------------------------------------------------
Chapter: 8
Title: The Neolithic Age: First Food Producers and Animal Keepers
--------------------------------------------------
Chapter: 9
Title: Chalcolithic Cultures
--------------------------------------------------
Chapter: 10
Title: Harappan Culture: Bronze Age Urbanization in the 

In [9]:
import re

positions = []

for chapter in chapters:

    pattern = re.escape(chapter)
    match = re.search(pattern, full_text_shorten.replace("\n"," "), re.IGNORECASE)

    if match:
        positions.append((chapter, match.start()))

In [10]:
chapters_text = []

for i in range(0,len(positions)):

    chapter_name, start = positions[i]

    if i < len(positions) - 1:
        end = positions[i + 1][1]
    else:
        end = len(full_text_shorten)

    text = full_text_shorten[start:end].strip()

    chapters_text.append({
        "chapter": chapter_name,
        "text": text
    })

In [11]:
# Create documents
from langchain_core.documents import Document

docs = []

for ch in chapters_text:
    docs.append(Document(page_content=ch['text'], metadata={"chapter": ch['chapter']}))

In [12]:
chapter_dict = {item: idx for idx, item in enumerate(chapters)}

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

from collections import defaultdict

chapter_counter = defaultdict(int)


for i, chunk in enumerate(chunks):
    chapter = chunk.metadata['chapter']
    chapter_counter[chapter] += 1
    chapter_num = chapter_dict[chapter] + 1

    chunk.metadata['chapter_chunk'] = chapter_counter[chapter]
    chunk.metadata['chunk_label'] = f"{chapter}_{chapter_counter[chapter]}"
    chunk.metadata['chunk_level'] = f"Chapter{chapter_num}_{chapter_counter[chapter]}"

    

In [14]:
for chunk in chunks:  # Print metadata of the first 5 chunks
    print(chunk.metadata)
    print("-" * 50)

{'chapter': 'The Significance of Ancient Indian History', 'chapter_chunk': 1, 'chunk_label': 'The Significance of Ancient Indian History_1', 'chunk_level': 'Chapter1_1'}
--------------------------------------------------
{'chapter': 'The Significance of Ancient Indian History', 'chapter_chunk': 2, 'chunk_label': 'The Significance of Ancient Indian History_2', 'chunk_level': 'Chapter1_2'}
--------------------------------------------------
{'chapter': 'The Significance of Ancient Indian History', 'chapter_chunk': 3, 'chunk_label': 'The Significance of Ancient Indian History_3', 'chunk_level': 'Chapter1_3'}
--------------------------------------------------
{'chapter': 'The Significance of Ancient Indian History', 'chapter_chunk': 4, 'chunk_label': 'The Significance of Ancient Indian History_4', 'chunk_level': 'Chapter1_4'}
--------------------------------------------------
{'chapter': 'The Significance of Ancient Indian History', 'chapter_chunk': 5, 'chunk_label': 'The Significance of An

In [15]:
for chunk in chunks:
    if chunk.metadata["chapter"] == 'From Ancient to Medieval':
        print(chunk.metadata['chapter_chunk'])
        print(chunk.page_content)

1
from ancient to medieval
times. Harsha governed his empire on the same lines as did the Guptas, but his
administration had become feudal and decentralized. It is stated that Harsha had
100,000 horses and 60,000 elephants. This appears astonishing because the
Mauryas, who ruled over virtually the entire country except the deep south,
maintained only 30,000 cavalry and 9000 elephants. Harsha could have had a
larger army only if he was in a position to mobilize the support of all his
feudatories in the time of war. Evidently every feudatory contributed his quota of
foot soldiers and horses, and thus enormously added to the imperial army. The vast numbers of the imperial army suggests a great increase in population.
Land grants continued to be made to priests for special services rendered to
the state. More importantly, Harsha is credited with the grant of land to the
officers by issuing charters. These grants allowed the same concessions to priests
2
officers by issuing charters. These 

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
vectorstore = FAISS.from_documents(chunks, embeddings)


/var/folders/1f/dc6nbcvj0yzdh4mpnz5f_3l00000gn/T/ipykernel_87635/2296132079.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5962.20it/s]


In [17]:
query = "What is the largest Buddhist temple in the world?"

results = vectorstore.similarity_search(query, k=5)

for r in results:
    print(r.metadata)
    print(r.page_content)
    print("-"*50)

{'chapter': 'From Ancient to Medieval', 'chapter_chunk': 59, 'chunk_label': 'From Ancient to Medieval_59', 'chunk_level': 'Chapter31_59'}
of a happy blending of both Indian and indigenous elements. It is astonishing
that the greatest Buddhist temple is to be found not in India but in Borobudur in
Java. Considered to be the largest Buddhist temple in the world, it was
constructed in the eighth century, and 436 images of the Buddha engraved on it
illustrate his life.
The temple of Ankor Vat in Cambodia is larger than that of Borobudur.
Although this temple dates to medieval times, it can be compared to the best
artistic achievements of the Egyptians and Greeks. The stories of the Ramayana
and Mahabharata are narrated in relief on the walls of the temple. The story of
the Ramayana is so popular in Indonesia that many folk plays based on it are
performed. The language of Indonesia, Bahasa Indonesia, contains numerous
Sanskrit words.
Cultural Give and Take
With regard to sculpture, the head

In [74]:

query = "What is the largest Buddhist temple in the world?"
from langsmith import traceable
retriever = vectorstore.as_retriever(
search_type="similarity",
search_kwargs={"k": 5}
)


@traceable
def generate_answer(query):
        # results = vectorstore.similarity_search(query, k=5)

        results = retriever.invoke(query)

        context = "\n".join([r.page_content for r in results])

        prompt = f"""
        You are a helpful teacher of Indian history. Answer the query ONLY based on provided context.
        Also put next line after every 100 characters in your answer. 
        If the answer is not present in the context, say "I don't know".
        Keep the output concise and suitable for a academic scholar in history.
        Context:{context}
        Query: {query}
        Answer:
        """

        from openai import OpenAI
        from langchain_openai import ChatOpenAI

        # client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        llm = ChatOpenAI(model='gpt-4o-mini')
        response = llm.invoke(input=prompt,temperature=0.5)
        # response = client.responses.create(model='gpt-4o-mini',
                # input = prompt,
                # temperature=0.4
                # # max_output_tokens=500,
                # )
        
        return response.content




In [50]:
# test langsmith tracing
answer = generate_answer("What is the largest Buddhist temple in the world?")

# Evaluation of LLM
## First part evaluation of Retrieval:
### 1. Recall@K
### 2. MRR (Mean Reciprocal Rank)

In [20]:
# Read the golden answer from the file
import pandas as pd
golden_df = pd.read_excel("data/golden_data_ancient.xlsx")
# golden_df =  #.set_index('Question')
golden_df['Gold_Chunk'] = golden_df['Gold_Chunk'].str.split(',')  # Strip comma from the golden chunk
display(golden_df.head())

,ID,Question,Ground Truth Chunk,Reference Answer,Gold_Chunk
0,1,Why is it important to study the history of An...,Chapter_1,The study of ancient Indian history is importa...,[Chapter1_1]
1,2,What was the role of Sir William Jones?,Chapter_2,Sir William Jones was a civil servant of East ...,"[Chapter2_1, Chapter2_2]"
2,3,How did the Rationalist approach the History?,Chapter_2,A section of the Indian scholars tries to refo...,"[Chapter2_7, Chapter2_8, Chapter2_17]"
3,4,Who was Ramakrishna Gopal Bhandarkar and what ...,Chapter_2,Ramakrishna Gopal Bhandarkar was a Indian Hist...,"[Chapter2_8, Chapter2_9]"
4,5,How are prehistoric sites are different from h...,Chapter_3,There are several difference between prehistor...,[Chapter3_1]


In [21]:
# Extract the retrieved chunks and the ranks from the retrieval results (Sample)

query = "What was the role of Sir William Jones?"

results = retriever.invoke(query, k=5)

for r in results:
    print(r.metadata)
    print(r.page_content)
    print("-"*50)


{'chapter': 'Modern Historians of Ancient India', 'chapter_chunk': 2, 'chunk_label': 'Modern Historians of Ancient India_2', 'chunk_level': 'Chapter2_2'}
until the eighteenth century, culminated in the establishment in Calcutta in 1784
of the Asiatic Society of Bengal. It was set up by a civil servant of the East India
Company, Sir William Jones (1746–94). He was the first to suggest that Sanskrit,
Latin, and Greek belonged to the same family of languages. He also translated
the play known as the Abhijnanashakuntalam into English in 1789; the
Bhagvadgita, the most popular Hindu religious text was translated into English
by Wilkins in 1785. The Bombay Asiatic Society was set up in 1804, and the
Asiatic Society of Great Britain was set up in London in 1823. William Jones
emphasized that originally the European languages were very similar to Sanskrit
and the Iranian language. This enthused European countries such as Germany,
France, and Russia, to foster Indological studies. During the fi

In [ ]:
def rag_app(inputs):
    query = inputs["question"]

    docs = retriever.get_relevant_documents(query)

    return {
        "answer": llm.invoke(query),
        "retrieved_chunks": [d.page_content for d in docs]
    }

In [24]:
golden_df.head()

,ID,Question,Ground Truth Chunk,Reference Answer,Gold_Chunk
0,1,Why is it important to study the history of An...,Chapter_1,The study of ancient Indian history is importa...,[Chapter1_1]
1,2,What was the role of Sir William Jones?,Chapter_2,Sir William Jones was a civil servant of East ...,"[Chapter2_1, Chapter2_2]"
2,3,How did the Rationalist approach the History?,Chapter_2,A section of the Indian scholars tries to refo...,"[Chapter2_7, Chapter2_8, Chapter2_17]"
3,4,Who was Ramakrishna Gopal Bhandarkar and what ...,Chapter_2,Ramakrishna Gopal Bhandarkar was a Indian Hist...,"[Chapter2_8, Chapter2_9]"
4,5,How are prehistoric sites are different from h...,Chapter_3,There are several difference between prehistor...,[Chapter3_1]


In [25]:
from tqdm import tqdm
evaluation_results = []

for _, row in tqdm(golden_df.iterrows()):

    query = row["Question"]
    gold_chunks = row["Gold_Chunk"]

    results = retriever.invoke(query, k=5)

    retrieved_chunks = [
        doc.metadata["chunk_level"]
        for doc in results
    ]

    metrics = evaluate_retrieval(
        gold_chunks,
        retrieved_chunks
    )

    evaluation_results.append({
        "Question": query,
        "Gold_Chunks": gold_chunks,
        "Retrieved_Chunks": retrieved_chunks,
        "Recall@1": metrics["Recall@1"],
        "Recall@2": metrics["Recall@2"],
        "Recall@3": metrics["Recall@3"],
        "Recall@5": metrics["Recall@5"],
        "RR": metrics["RR"]
    })

50it [00:04, 10.97it/s]


In [26]:
eval_df = pd.DataFrame(evaluation_results)

print("===== Retrieval Metrics =====")
print(f"Recall@1 : {eval_df['Recall@1'].mean():.3f}")
print(f"Recall@2 : {eval_df['Recall@2'].mean():.3f}")
print(f"Recall@3 : {eval_df['Recall@3'].mean():.3f}")
print(f"Recall@5 : {eval_df['Recall@5'].mean():.3f}")
print(f"MRR       : {eval_df['RR'].mean():.3f}")

===== Retrieval Metrics =====
Recall@1 : 0.340
Recall@2 : 0.477
Recall@3 : 0.530
Recall@5 : 0.737
MRR       : 0.616


In [27]:
failures = eval_df[eval_df["Recall@5"] < 0.5]

failures[
    [
        "Question",
        "Gold_Chunks",
        "Retrieved_Chunks",
        "Recall@5"
    ]
]

,Question,Gold_Chunks,Retrieved_Chunks,Recall@5
2,How did the Rationalist approach the History?,"[Chapter2_7, Chapter2_8, Chapter2_17]","[Chapter33_7, Chapter2_17, Chapter2_14, Chapte...",0.333333
10,How ancient Indians view the rivers?,[Chapter5_10],"[Chapter1_10, Chapter4_10, Chapter2_16, Chapte...",0.000000
12,Tell me about the Indo-Aryan language group,"[Chapter6_7, Chapter6_8]","[Chapter6_1, Chapter6_9, Chapter12_54, Chapter...",0.000000
16,How was the agriculture of Neolithic setllers?,"[Chapter8_2, Chapter8_5, Chapter8_9]","[Chapter8_11, Chapter12_85, Chapter8_9, Chapte...",0.333333
18,What was the food of the Chalcolithc communities?,"[Chapter9_7, Chapter9_8]","[Chapter12_93, Chapter9_21, Chapter12_108, Cha...",0.000000
22,Role of War Chariot in Indo-European culture,[Chapter12_172],"[Chapter12_174, Chapter12_193, Chapter12_194, ...",0.000000
29,Who was Gautama Buddha?,"[Chapter14_21, Chapter14_20, Chapter_22]","[Chapter14_21, Chapter14_22, Chapter14_11, Cha...",0.333333
36,What were the names of the republic states dur...,[Chapter17_24],"[Chapter31_11, Chapter24_6, Chapter28_2, Chapt...",0.000000
39,What was the official language of the Satavaha...,[Chapter21_21],"[Chapter31_14, Chapter28_4, Chapter21_19, Chap...",0.000000
42,What was the capital of Harsha?,[Chapter26_25],"[Chapter26_27, Chapter26_24, Chapter15_3, Chap...",0.000000


## LLM as Judge for evaluation of Answer


In [73]:
from langchain_openai import ChatOpenAI
from langsmith import traceable
import json
import re


def safe_json_parse(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.S)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass

        return {
            "correctness": 0,
            "faithfulness": 0,
            "relevance": 0,
            "completeness": 0,
            "overall_score": 0,
            "reason": "parse_failed"
        }


@traceable(name="llm_judge")
def evaluate_answer(
    question: str,
    gold_answer: str,
    generated_answer: str,
    retrieved_context: str,
    model: str = "gpt-4o-mini"   # safer default
):

    prompt = f"""
You are an expert RAG evaluator.
As per the following guidelines:
1. correctness (1-5)
Measures whether the generated answer is factually correct compared to the gold answer and retrieved context.
2. faithfulness (1-5)
Measures whether the answer is strictly grounded in the retrieved context (no hallucination).
3. relevance (1-5)
Measures how well the answer addresses the user’s question.
4. completeness (1-5)
Measures whether the answer covers all key aspects needed to fully answer the question.
5. overall_score (float)
Weighted or holistic score (0–5) reflecting overall quality across all dimensions.


Return ONLY valid JSON:

{{
    "correctness": 1-5,
    "faithfulness": 1-5,
    "relevance": 1-5,
    "completeness": 1-5,
    "overall_score": float,
    "reason": "short explanation"
}}

Question:
{question}

Reference Answer:
{gold_answer}

Retrieved Context:
{retrieved_context}

Generated Answer:
{generated_answer}
"""

    llm = ChatOpenAI(model=model, temperature=0)

    response = llm.invoke(prompt)

    return safe_json_parse(response.content)

In [55]:
# test with one
question = "Write detailed description of the Mohenjo-daro bath"
golden_answer = golden_df.loc[golden_df['Question'] == question, 'Reference Answer'].values[0]
generated_answer = generate_answer(question)
context_results = retriever.invoke(question, k=5)
retrieved_context = "\n".join([r.page_content for r in context_results])

metrics = evaluate_answer(
    question=question,
    gold_answer=golden_answer,
    generated_answer=generated_answer,
    retrieved_context=retrieved_context
)



<REDACTED>


In [ ]:

evaluation_results = []

for idx, row in golden_df.iterrows():

    question = row["Question"]
    golden_answer = row["Reference Answer"]

    # Generate RAG answer
    generated_answer,retrieved_context = generate_answer(question)

    # Adjust if your chain returns a dict
    if isinstance(generated_answer, dict):
        generated_answer = generated_answer["answer"]

    # Retrieve context used for evaluation
    docs = retriever.invoke(question, k=5)

    retrieved_context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # LLM Judge
    metrics = evaluate_answer(
        question=question,
        gold_answer=golden_answer,
        generated_answer=generated_answer,
        retrieved_context=retrieved_context
    )

    evaluation_results.append({
        "Question": question,
        "Correctness": metrics["correctness"],
        "Faithfulness": metrics["faithfulness"],
        "Relevance": metrics["relevance"],
        "Completeness": metrics["completeness"],
        "Overall": metrics["overall_score"],
        "Reason": metrics["reason"]
    })

    print(
        f"{idx+1}/{len(golden_df)} | "
        f"Overall={metrics['overall_score']}"
    )

# Create results dataframe
eval_df = pd.DataFrame(evaluation_results)

1/50 | Overall=4.75
2/50 | Overall=5.0
3/50 | Overall=1.0
4/50 | Overall=4.25
5/50 | Overall=4.0
6/50 | Overall=4.75
7/50 | Overall=4.0
8/50 | Overall=4.75
9/50 | Overall=1.0
10/50 | Overall=4.75
11/50 | Overall=4.75
12/50 | Overall=3.5
13/50 | Overall=3.5
14/50 | Overall=4.75
15/50 | Overall=5.0
16/50 | Overall=5.0
17/50 | Overall=4.75
18/50 | Overall=4.25
19/50 | Overall=4.25
20/50 | Overall=5.0
21/50 | Overall=1.0
22/50 | Overall=5.0
23/50 | Overall=4.25
24/50 | Overall=4.5
25/50 | Overall=4.5
26/50 | Overall=4.75
27/50 | Overall=3.75
28/50 | Overall=4.5
29/50 | Overall=5.0
30/50 | Overall=5.0
31/50 | Overall=4.75
32/50 | Overall=5.0
33/50 | Overall=5.0
34/50 | Overall=4.75
35/50 | Overall=1.0
36/50 | Overall=4.0
37/50 | Overall=2.0
38/50 | Overall=4.75
39/50 | Overall=5.0
40/50 | Overall=5.0
41/50 | Overall=5.0
42/50 | Overall=4.25
43/50 | Overall=1.0
44/50 | Overall=4.5
45/50 | Overall=4.75
46/50 | Overall=5.0
47/50 | Overall=5.0
48/50 | Overall=4.5
49/50 | Overall=5.0
50/50 | Ove

In [65]:
eval_df

,Question,Correctness,Faithfulness,Relevance,Completeness,Overall,Reason
0,Why is it important to study the history of An...,5,5,5,4,4.75,The answer accurately captures the main reason...
1,What was the role of Sir William Jones?,5,5,5,5,5.00,The answer accurately reflects the retrieved c...
2,How did the Rationalist approach the History?,1,1,1,1,1.00,Response is blank ('I don't know') and does no...
3,Who was Ramakrishna Gopal Bhandarkar and what ...,5,4,5,3,4.25,Accurately describes who Bhandarkar was and hi...
4,How are prehistoric sites are different from h...,4,4,5,3,4.00,Accurate core distinctions (prehistory vs hist...
5,How history of climate and vegetations are stu...,5,5,4,5,4.75,Core claim (pollen analysis for climate/vegeta...
6,Summarize Sangam Literature as a historical so...,4,4,5,3,4.00,Covers core facts (Sangam college under royal ...
7,Describe the creation of India as a geographic...,5,5,5,4,4.75,Covers the main creation timeline and geograph...
8,What are the major rivers in India?,1,1,1,1,1.00,The answer provides no information about the m...
9,Where are Copper mines located in India?,5,5,5,4,4.75,Accurately states the main locations (Chhotana...


In [66]:
eval_df[[
    "Correctness",
    "Faithfulness",
    "Relevance",
    "Completeness",
    "Overall"
]].mean()

Correctness     4.200
Faithfulness    4.420
Relevance       4.420
Completeness    3.800
Overall         4.205
dtype: float64

In [69]:
summary = {
    "Correctness": eval_df["Correctness"].mean(),
    "Faithfulness": eval_df["Faithfulness"].mean(),
    "Relevance": eval_df["Relevance"].mean(),
    "Completeness": eval_df["Completeness"].mean(),
    "Overall": eval_df["Overall"].mean(),
    # "Pass_Rate": eval_df["Pass"].mean()
}

summary

{'Correctness': np.float64(4.2),
 'Faithfulness': np.float64(4.42),
 'Relevance': np.float64(4.42),
 'Completeness': np.float64(3.8),
 'Overall': np.float64(4.205)}

In [3]:
from langsmith import traceable

@traceable
def generate_answer(question):
    return "test answer"

generate_answer("test")

'test answer'

# Langsmith Evaluation

In [56]:
def rag_app(inputs: dict):
    question = inputs["Question"]

    context_results = retriever.invoke(question, k=5)
    retrieved_context = "\n\n".join(
        doc.page_content for doc in context_results
    )
    answer = generate_answer(question)

    return {
        "answer": answer,
        "context": retrieved_context
    }

In [57]:
JUDGE_CACHE = {}



# ----------------------------
# 1. CACHE METRICS (IMPORTANT OPTIMIZATION)
# ----------------------------
def get_metrics_once(run, example):

    cache_key = run.id

    if cache_key not in JUDGE_CACHE:

        JUDGE_CACHE[cache_key] = evaluate_answer(
            question=example.inputs["Question"],
            gold_answer=example.outputs["Reference Answer"],
            generated_answer=run.outputs["answer"],
            retrieved_context=run.outputs["context"]
        )

    return JUDGE_CACHE[cache_key]

# ----------------------------
# 2. LANGSMITH EVALUATORS (NO LLM CALLS HERE)
# ----------------------------
def correctness_evaluator(run, example):
    m = get_metrics_once(run, example)
    return {"key": "correctness", "score": m["correctness"]}


def faithfulness_evaluator(run, example):
    m = get_metrics_once(run, example)
    return {"key": "faithfulness", "score": m["faithfulness"]}


def relevance_evaluator(run, example):
    m = get_metrics_once(run, example)
    return {"key": "relevance", "score": m["relevance"]}


def completeness_evaluator(run, example):
    m = get_metrics_once(run, example)
    return {"key": "completeness", "score": m["completeness"]}


In [76]:
# ----------------------------
# 3. RUN LANGSMITH EVALUATION
# ----------------------------
from langsmith import Client

client = Client()

results = client.evaluate(
    rag_app,
    data="ancient_rag",
    evaluators=[
        correctness_evaluator,
        faithfulness_evaluator,
        relevance_evaluator,
        completeness_evaluator,
    ],
    experiment_prefix="rag-v2-highT"
)

View the evaluation results for experiment: 'rag-v2-highT-6636583c' at:
https://smith.langchain.com/o/a7df2e3a-7089-4e7a-b50e-dfaf56d72201/datasets/77502e77-588a-4664-a54d-a1f4a04d7bcf/compare?selectedSessions=0a072647-f610-43aa-a67a-65951548812d




50it [03:43,  4.47s/it]


In [61]:
print("JUDGE OUTPUT:", m)

JUDGE OUTPUT: <re.Match object; span=(12730, 12768), match='33.\nLegacy in Science and Civilization'>


In [77]:
from langsmith import traceable

@traceable(name="retrieval")
def retrieve_docs(query):
    docs = retriever.invoke(query, k=5)
    return docs

In [96]:
def rag_app_ret(inputs):
    query = inputs["Question"]

    docs = retriever.invoke(query)
    answer = generate_answer(query)

    

    return {
        "answer": answer,
        "retrieved_chunks": [d.page_content for d in docs],
        "chunk_level": [d.metadata['chunk_level'] for d in docs]
    }

In [97]:

@traceable(name="retrieval_metrics")
def evaluate_retrieval(run, example):

    retrieved_chunks = set(run.outputs["chunk_level"])

    gold_chunks = set(example.outputs["gold_chunks"])

    metrics = {}

    # Recall@K
    for k in [1, 2, 3, 5]:
        top_k = set(retrieved_chunks[:k])
        hits = len(gold_chunks & top_k)
        metrics[f"Recall@{k}"] = hits / len(gold_chunks)

    # Reciprocal Rank (MRR per query)
    rr = 0.0
    for rank, chunk in enumerate(retrieved_chunks, start=1):
        if chunk in gold_chunks:
            rr = 1 / rank
            break

    metrics["RR"] = rr

    return metrics

In [110]:
def evaluate_retrieval(run, example):

    retrieved = run.outputs["chunk_level"]
    gold = example.outputs["Gold_Chunk"]
    # print(retrieved)
    # print(gold)

    retrieved_set = set(retrieved)
    gold_set = {x.strip() for x in gold.split(",")}

    intersection = retrieved_set & gold_set
    # print(intersection)

    # Recall@K
    recall = (
        len(intersection) / len(gold_set)
        if gold_set else 0
    )

    # Precision@K
    precision = (
        len(intersection) / len(retrieved_set)
        if retrieved_set else 0
    )

    # F1@K
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    # Hit Rate
    hit_rate = 1 if len(intersection) > 0 else 0

    # MRR
    mrr = 0
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in gold_set:
            mrr = 1 / rank
            break

    return {
        "results": [
            {"key": "Recall@K", "score": recall},
            {"key": "Precision@K", "score": precision},
            {"key": "F1@K", "score": f1},
            {"key": "HitRate@K", "score": hit_rate},
            {"key": "MRR", "score": mrr},
        ]
    }

In [111]:
from langsmith import evaluate

results = evaluate(
    rag_app_ret,
    data="ancient_rag",
    evaluators=[evaluate_retrieval],
    experiment_prefix="rag-retrieval-metrics"
)

View the evaluation results for experiment: 'rag-retrieval-metrics-1aa41e53' at:
https://smith.langchain.com/o/a7df2e3a-7089-4e7a-b50e-dfaf56d72201/datasets/77502e77-588a-4664-a54d-a1f4a04d7bcf/compare?selectedSessions=7b40d6dd-4b63-4602-9da0-bfcc2e852a12




50it [02:24,  2.89s/it]
